In [ ]:
#导包
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams['font.sans-serif'] = ['SimHei']
plt.rcParams['axes.unicode_minus'] = False
import jieba
from collections import Counter
from wordcloud import WordCloud

In [ ]:
#1.导入文件
df = pd.read_csv(r'D:\数据分析实战\data_analyse\UserBehavior--1.csv', encoding='gb18030')
print(df.head())
print(df.shape)
print(df.info())
print(df.describe())


In [ ]:
# 2.数据清洗-----检查重复值
df.duplicated().sum()

In [ ]:
#检查缺失值
df.isnull().sum()

In [ ]:
# 地址缺失用“未知”填充
df['address'] = df['address'].fillna('未知')

# 评论缺失不用删
df['comment'] = df['comment'].fillna('无评论')

In [ ]:
# 转换时间戳，异常值变NaT
df['timestamp'] = pd.to_datetime(df['timestamp'], unit='s', errors='coerce')
#删除时间为空的行
df = df.dropna(subset=['timestamp'])

In [ ]:
#拆时间维度
df['date'] = df['timestamp'].dt.date
df['hour'] = df['timestamp'].dt.hour
df['month'] = df['timestamp'].dt.month
df['weekday'] = df['timestamp'].dt.day_name()

In [ ]:
#goods_id字段处理
df[df['goods_id'] < 0]     #之前的数据处理正好清除goods_id负值异常，结果为0


In [ ]:
#price字段处理
df[(df['behavior']=='buy') & (df['price']==0)]  #11, 15
df = df[~((df['behavior']=='buy') & (df['price']==0))]

In [ ]:
#amount字段处理
df[(df['behavior']=='buy') & (df['amount']==0)]  #结果为0

In [ ]:
#统一字段行为
df['behavior'].value_counts() #正常

In [ ]:
#新增销售额字段
df['sales'] = df['price'] * df['amount']

df.to_csv('clean_jd_data.csv', index=False, encoding='utf-8-sig')

In [ ]:
#3.用户行为分析----用户行为柱状图
behavior_count = df['behavior'].value_counts()
print(behavior_count)

behavior_count.plot(kind='bar')
plt.title('User Behavior Distribution')
plt.xlabel('Behavior')
plt.ylabel('Count')
plt.show()

In [ ]:
#用户行为饼状图
behavior_count.plot(
    kind='pie',
    autopct='%1.1f%%',
    figsize=(8,8)
)

plt.ylabel('')
plt.title('Behavior Ratio')
plt.show()

In [ ]:
#转化漏斗分析
pv = behavior_count['pv']
cart = behavior_count['cart']
fav = behavior_count['fav']
buy = behavior_count['buy']

cart_rate = cart / pv
fav_rate = fav / pv
buy_rate = buy / pv

print("加购率：", cart_rate)
print("收藏率：", fav_rate)
print("购买率：", buy_rate)

In [ ]:
#4.时间趋势分析----日行为趋势
daily_behavior = df.groupby('date').size()
print(daily_behavior)

daily_behavior.plot(figsize=(12,6))
plt.title('Daily User Activity')
plt.xlabel('Date')
plt.ylabel('Count')
plt.show()

In [ ]:
#小时行为趋势
hour_behavior = df.groupby('hour').size()
print(hour_behavior)

hour_behavior.plot(figsize=(12,6))
plt.title('Hourly Activity')
plt.xlabel('Hour')
plt.ylabel('Count')
plt.show()

In [ ]:
#5.商品分析----TOP销量商品分析
top_goods = df[df['behavior']=='buy'].groupby('goods_id')['amount'].sum().sort_values(ascending=False).head(10)
print(top_goods)

top_goods.plot(kind='bar', figsize=(12,6))
plt.title('Top 10 Goods by Sales Volume')
plt.xlabel('Goods ID')
plt.ylabel('Sales Volume')
plt.show()

In [ ]:
#TOP销售额商品分析
top_sales_goods = df.groupby('goods_id')['sales'].sum().sort_values(ascending=False).head(10)
print(top_sales_goods)

top_sales_goods.plot(kind='bar', figsize=(12,6))
plt.title('Top 10 Goods by Revenue')
plt.xlabel('Goods ID')
plt.ylabel('Revenue')
plt.show()

In [ ]:
#TOP热门品类分析
top_category = df['category_id'].value_counts().head(10)
print(top_category)

top_category.plot(kind='bar', figsize=(12,6))
plt.title('Top 10 Categories by Activity')
plt.xlabel('Category ID')
plt.ylabel('Count')
plt.show()

In [ ]:
#TOP销售品类分析
category_sales = df.groupby('category_id')['sales'].sum().sort_values(ascending=False).head(10)
print(category_sales)

category_sales.plot(kind='bar', figsize=(12,6))
plt.title('Top 10 Categories by Revenue')
plt.xlabel('Category ID')
plt.ylabel('Revenue')
plt.show()

In [ ]:
#6.用户价值分析----高消费用户分析
user_sales = df.groupby('user_id')['sales'].sum().sort_values(ascending=False).head(10)
print(user_sales)

user_sales.plot(kind='bar', figsize=(12,6))
plt.title('Top 10 Users by Revenue')
plt.xlabel('User ID')
plt.ylabel('Revenue')
plt.show()

In [ ]:
#购买次数最多用户
top_buy_users = df[df['behavior']=='buy']['user_id'].value_counts().head(10)
print(top_buy_users)

top_buy_users.plot(kind='bar', figsize=(12,6))
plt.title('Top 10 Users by Purchase Frequency')
plt.xlabel('User ID')
plt.ylabel('Purchase Count')
plt.show()

In [ ]:
#客单价分析
avg_order_value = df[df['behavior']=='buy']['sales'].sum() / len(df[df['behavior']=='buy'])
print(avg_order_value)

In [ ]:
#用户消费分层
user_total_sales = df.groupby('user_id')['sales'].sum()
print(user_total_sales.describe())

user_level = pd.cut(
    user_total_sales,
    bins=[0,50,200,1000,10000],
    labels=['低消费','中消费','高消费','超级用户']
)

print(user_level.value_counts())

In [ ]:
#7.RFM用户价值模型----筛选购买数据
buy_df = df[df['behavior']=='buy'].copy()

#构建R
snapshot_date = buy_df['timestamp'].max() + pd.Timedelta(days=1)
print(snapshot_date)

R = buy_df.groupby('user_id')['timestamp'].max()
R = (snapshot_date - R).dt.days
print(R.head())

In [ ]:
#构建F
F = buy_df.groupby('user_id').size()
print(F.head())

In [ ]:
#构建M
M = buy_df.groupby('user_id')['sales'].sum()
print(M.head())

In [ ]:
#合并RFM
rfm = pd.concat([R,F,M], axis=1)
rfm.columns = ['R','F','M']

In [ ]:
#构建用户标签
rfm['R_score'] = rfm['R'].apply(
    lambda x: 1 if x <= rfm['R'].median() else 0
)
rfm['F_score'] = rfm['F'].apply(
    lambda x: 1 if x >= rfm['F'].median() else 0
)
rfm['M_score'] = rfm['M'].apply(
    lambda x: 1 if x >= rfm['M'].median() else 0
)

rfm['RFM'] = (
    rfm['R_score'].astype(str) +
    rfm['F_score'].astype(str) +
    rfm['M_score'].astype(str)
)
print(rfm.head())
print(rfm['RFM'].value_counts())

In [ ]:
#建立用户标签映射
def rfm_label(x):
    if x == '111':
        return '核心价值用户'
    elif x == '110':
        return '高潜力用户'
    elif x == '101':
        return '高消费流失预警用户'
    elif x == '100':
        return '一般发展用户'
    elif x == '011':
        return '重点保持用户'
    elif x == '010':
        return '一般保持用户'
    elif x == '001':
        return '高消费沉睡用户'
    else:
        return '流失用户'

rfm['label'] = rfm['RFM'].apply(rfm_label)

label_count = rfm['label'].value_counts()
print(label_count)

In [ ]:
#画用户标签映射柱状图
label_count.plot(
    kind='bar',
    figsize=(12,6)
)

plt.title('RFM User Segmentation')
plt.xlabel('User Segment')
plt.ylabel('Count')
plt.show()

In [ ]:
#8.地区分析----地区活跃度分析
area_active = df['address'].value_counts().head(10)
print(area_active)

area_active.plot(kind='bar', figsize=(12,6))
plt.title('Top 10 Active Areas')
plt.xlabel('Area')
plt.ylabel('Activity Count')
plt.show()

In [ ]:
#地区销售额分析
area_sales = df.groupby('address')['sales'].sum().sort_values(ascending=False).head(10)
print(area_sales)

area_sales.plot(kind='bar', figsize=(12,6))
plt.title('Top 10 Areas by Revenue')
plt.xlabel('Area')
plt.ylabel('Revenue')
plt.show()

In [ ]:
#地区购买次数分析
area_buy = df[df['behavior']=='buy']['address'].value_counts().head(10)
print(area_buy)

area_buy.plot(kind='bar', figsize=(12,6))
plt.title('Top 10 Areas by Purchase Count')
plt.xlabel('Area')
plt.ylabel('Purchase Count')
plt.show()

In [ ]:
#9.设备分析----设备活跃度分析
device_active = df['device'].value_counts()
print(device_active)

device_active.plot(kind='bar', figsize=(12,6))
plt.title('Device Activity Distribution')
plt.xlabel('Device')
plt.ylabel('Count')
plt.show()

In [ ]:
#设备购买次数分析
device_buy = df[df['behavior']=='buy']['device'].value_counts()
print(device_buy)

device_buy.plot(kind='bar', figsize=(12,6))
plt.title('Device Purchase Distribution')
plt.xlabel('Device')
plt.ylabel('Purchase Count')
plt.show()

In [ ]:
#设备销售额分析
device_sales = df.groupby('device')['sales'].sum().sort_values(ascending=False)
print(device_sales)

device_sales.plot(kind='bar', figsize=(12,6))
plt.title('Device Revenue Distribution')
plt.xlabel('Device')
plt.ylabel('Revenue')
plt.show()

In [ ]:
#设备转化率分析
device_conversion = device_buy / device_active
print(device_conversion)

In [ ]:
#10.评论分析----评论数量分析
comment_df = df[df['comment'] != '无评论'].copy()
print(comment_df.shape)
comment_count = comment_df.shape[0]
print(comment_count)

In [ ]:
#高频词分析
#拼接文本
text = ''.join(comment_df['comment'].astype(str))
#分词
words = jieba.lcut(text)
print(words[:50])
#过滤停用词
stopwords = ['的','了','很','也','是','都',
    '一个','这个','就是','还是',
    '什么','没有','不是','知道']
words = [w for w in words if w not in stopwords and len(w) > 1]
#统计词频
word_count = Counter(words)
print(word_count.most_common(20))

In [ ]:
#词云图
wc = WordCloud(
    font_path='simhei.ttf',
    width=1000,
    height=600
)
wc.generate(' '.join(words))

plt.figure(figsize=(12,6))
plt.imshow(wc)
plt.axis('off')
plt.show()

In [ ]:
df.to_csv(
    'jd_analysis_final.csv',
    index=False,
    encoding='utf-8-sig'
)

In [ ]:
rfm.to_csv(
    'jd_rfm_result.csv',
    encoding='utf-8-sig'
)